In [13]:
import numpy as np
h_0 = np.array([[0], [0], [1]])
w_xh = np.array([[1], [0], [0]]) # Wxh shape (3,1)
w_hh = np.array([[0, 0, 0],
                 [1, 0, 0],
                 [0, 0, 1]]) # Whh shape (3,3)
w_hy = np.array([1, 1, -1]) # Why shape (1,3)

# x_in = [1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1]
x_in = [1,1,1,1,1,1,1,1,1]

x_seq = np.array([[x] for x in x_in]) # Format properly for matmul ops

h_t_prev = h_0

def relu(x):
    return np.maximum(0, x)

y_seq = []
for t, x in enumerate(x_seq):
    h_t = relu(w_hh @ h_t_prev + (w_xh @ x).reshape(3,1))
    y_t = relu(w_hy @ h_t)
    y_seq.append(y_t)
    h_t_prev = h_t

print("Inputs: ", [int(x) for x in x_seq.flatten()])
print("Outputs:", [int(y[0]) for y in y_seq])

Inputs:  [1, 1, 1, 1, 1, 1, 1, 1, 1]
Outputs: [0, 1, 1, 1, 1, 1, 1, 1, 1]


In [16]:
"""
Minimal character-level Vanilla RNN model. Written by Andrej Karpathy (@karpathy)
BSD License
"""


# data I/O
data = open('input.txt', 'r').read() # should be simple plain text file
chars = list(set(data))
data_size, vocab_size = len(data), len(chars)
print ('data has %d characters, %d unique.' % (data_size, vocab_size))
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

# hyperparameters
hidden_size = 100 # size of hidden layer of neurons
seq_length = 25 # number of steps to unroll the RNN for
learning_rate = 1e-1

# model parameters
Wxh = np.random.randn(hidden_size, vocab_size)*0.01 # input to hidden
Whh = np.random.randn(hidden_size, hidden_size)*0.01 # hidden to hidden
Why = np.random.randn(vocab_size, hidden_size)*0.01 # hidden to output
bh = np.zeros((hidden_size, 1)) # hidden bias
by = np.zeros((vocab_size, 1)) # output bias

def lossFun(inputs, targets, hprev):
  """
  inputs,targets are both list of integers.
  hprev is Hx1 array of initial hidden state
  returns the loss, gradients on model parameters, and last hidden state
  """
  xs, hs, ys, ps = {}, {}, {}, {}
  hs[-1] = np.copy(hprev)
  loss = 0
  # forward pass
  for t in range(len(inputs)):
    xs[t] = np.zeros((vocab_size,1)) # encode in 1-of-k representation
    xs[t][inputs[t]] = 1
    hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh) # hidden state
    ys[t] = np.dot(Why, hs[t]) + by # unnormalized log probabilities for next chars
    ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t])) # probabilities for next chars
    loss += -np.log(ps[t][targets[t],0]) # softmax (cross-entropy loss)
  # backward pass: compute gradients going backwards
  dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
  dbh, dby = np.zeros_like(bh), np.zeros_like(by)
  dhnext = np.zeros_like(hs[0])
  for t in reversed(range(len(inputs))):
    dy = np.copy(ps[t])
    dy[targets[t]] -= 1 # backprop into y. see http://cs231n.github.io/neural-networks-case-study/#grad if confused here
    dWhy += np.dot(dy, hs[t].T)
    dby += dy
    dh = np.dot(Why.T, dy) + dhnext # backprop into h
    dhraw = (1 - hs[t] * hs[t]) * dh # backprop through tanh nonlinearity
    dbh += dhraw
    dWxh += np.dot(dhraw, xs[t].T)
    dWhh += np.dot(dhraw, hs[t-1].T)
    dhnext = np.dot(Whh.T, dhraw)
  for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
    np.clip(dparam, -5, 5, out=dparam) # clip to mitigate exploding gradients
  return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

def sample(h, seed_ix, n):
  """ 
  sample a sequence of integers from the model 
  h is memory state, seed_ix is seed letter for first time step
  """
  x = np.zeros((vocab_size, 1))
  x[seed_ix] = 1
  ixes = []
  for t in range(n):
    h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)
    y = np.dot(Why, h) + by
    p = np.exp(y) / np.sum(np.exp(y))
    ix = np.random.choice(range(vocab_size), p=p.ravel())
    x = np.zeros((vocab_size, 1))
    x[ix] = 1
    ixes.append(ix)
  return ixes

n, p = 0, 0
mWxh, mWhh, mWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
mbh, mby = np.zeros_like(bh), np.zeros_like(by) # memory variables for Adagrad
smooth_loss = -np.log(1.0/vocab_size)*seq_length # loss at iteration 0
while True:
  # prepare inputs (we're sweeping from left to right in steps seq_length long)
  if p+seq_length+1 >= len(data) or n == 0: 
    hprev = np.zeros((hidden_size,1)) # reset RNN memory
    p = 0 # go from start of data
  inputs = [char_to_ix[ch] for ch in data[p:p+seq_length]]
  targets = [char_to_ix[ch] for ch in data[p+1:p+seq_length+1]]

  # sample from the model now and then
  if n % 100 == 0:
    sample_ix = sample(hprev, inputs[0], 200)
    txt = ''.join(ix_to_char[ix] for ix in sample_ix)
    print ('----\n %s \n----' % (txt, ))

  # forward seq_length characters through the net and fetch gradient
  loss, dWxh, dWhh, dWhy, dbh, dby, hprev = lossFun(inputs, targets, hprev)
  smooth_loss = smooth_loss * 0.999 + loss * 0.001
  if n % 100 == 0: print ('iter %d, loss: %f' % (n, smooth_loss)) # print progress
  
  # perform parameter update with Adagrad
  for param, dparam, mem in zip([Wxh, Whh, Why, bh, by], 
                                [dWxh, dWhh, dWhy, dbh, dby], 
                                [mWxh, mWhh, mWhy, mbh, mby]):
    mem += dparam * dparam
    param += -learning_rate * dparam / np.sqrt(mem + 1e-8) # adagrad update

  p += seq_length # move data pointer
  n += 1 # iteration counter 

data has 632 characters, 42 unique.
----
 bmI;atWy,MgyrMâpcxIcMRN :wMIR,Ir;b'?NiynTTS™oy?vhhTeI€ eycpaf.xoph:':i™mSkRebBTxvNy.™lecm'mg™'€,cdhWaksa;i?dtTbxklS€bs€vehn;.o,teWn™uv'bmBdcsdMlfSimvba;cnyokm?nSs Agydmn :W,wAbyu'A;xo;A'Wv,.fpAlIxi'd, 
----
iter 0, loss: 93.441737
----
 nospa't rpe dyaeT vo taaMtSoacau™uee SpoeheâyIlsd Woe egMla  niTmNnc ,sRloaitn  s esâ  aean hfwlâwhoui tternl thhm ylsog nyeede w Neaag:ssegi akt ih  er™i™ gs  :di ytaeâgtiann iuvns ,hklr isws seae  s 
----
iter 100, loss: 93.701194
----
  alvthgr mrâ ih snu,verl euis; i eylA eeoa x ingigelt,gesela ™a' usc  t rafef  S€et  pre d g™elâ€:ien™sbau i ,ea iis eNcy tonS€erdla io dan f  em  esentâenegthila â,e   das sr,roinnbomos rhanteiiar ou 
----
iter 200, loss: 92.123789
----
 hanowfgp iil tnavpI'an.n rlg  svanys ardBpafefgreao cAigmiriitwhaceN lSd mg lrg linW esm te amg  tm, lnllrosraâ lSi âonilâNM lAt  o tha;ma?âdrs™ m™uh khi sgoah'tanamaor tm,iindhhes as, famesrg aiolaâd 
----
iter 300, loss: 90.048845
----
 nalc

KeyboardInterrupt: 